In [159]:
import pandas as pd
import numpy as np

In [160]:
# Load BRFSS 2024 Dataset
df = pd.read_sas("../data/raw/LLCP2024.XPT")

print("Dataset Shape:", df.shape)

Dataset Shape: (457670, 301)


In [161]:
# Dataset Overview
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 457670 entries, 0 to 457669
Columns: 301 entries, _STATE to _AIDTST4
dtypes: float64(296), object(5)
memory usage: 1.0+ GB


In [162]:
# Confirm existence of target variable
'MEDCOST1' in df.columns

True

In [163]:
df['MEDCOST1'].value_counts(dropna=False)

MEDCOST1
2.0    412634
1.0     43363
7.0      1229
9.0       438
NaN         6
Name: count, dtype: int64

In [164]:
# Target variable proportions
df["MEDCOST1"].value_counts(normalize=True, dropna=False) * 100

MEDCOST1
2.0    90.159722
1.0     9.474731
7.0     0.268534
9.0     0.095702
NaN     0.001311
Name: proportion, dtype: float64

In [165]:
# Create dataframe of all column names
columns_df = pd.DataFrame({
    "Variable": df.columns
})
columns_df.head(20)

,Variable
0,_STATE
1,FMONTH
2,IDATE
3,IMONTH
4,IDAY
5,IYEAR
6,DISPCODE
7,SEQNO
8,_PSU
9,CTELENM1


# Candidate Predictor Variables

## Target Variable
- MEDCOST1 — Was there a time in the past 12 months when you needed to see a doctor but could not because of cost?

---

## Demographics
- SEXVAR — Sex of respondent
- MARITAL — Marital status
- EDUCA — Highest grade or year of school completed
- RENTHOM1 — Home ownership status (own or rent)
- _AGE80 — Imputed age in years (top coded at 80)
- _RACEGR3 — Five-level race/ethnicity classification

---

## Socioeconomic
- INCOME3 — Annual household income from all sources
- EMPLOY1 — Current employment status
- VETERAN3 — Ever served on active duty in the United States Armed Forces

---

## Healthcare Access
- PRIMINS2 — Current primary source of health care coverage
- PERSDOC3 — Personal doctor or health care provider status
- CHECKUP1 — Time since last routine medical checkup

---

## General Health Status
- GENHLTH — Self-reported general health
- PHYSHLTH — Number of days physical health was not good during the past 30 days
- MENTHLTH — Number of days mental health was not good during the past 30 days
- POORHLTH — Number of days poor physical or mental health limited usual activities

---

## Chronic Conditions
- CVDINFR4 — Ever told you had a heart attack (myocardial infarction)
- CVDCRHD4 — Ever told you had angina or coronary heart disease
- CVDSTRK3 — Ever told you had a stroke
- ASTHMA3 — Ever told you had asthma
- ASTHNOW — Still have asthma
- CHCSCNC1 — Ever told you had non-melanoma skin cancer
- CHCOCNC1 — Ever told you had melanoma or another type of cancer
- CHCCOPD3 — Ever told you had COPD, emphysema, or chronic bronchitis
- ADDEPEV3 — Ever told you had a depressive disorder
- CHCKDNY2 — Ever told you had kidney disease (excluding kidney stones, bladder infections, or incontinence)

---

## Health Behaviors
- EXERANY2 — Participated in physical activities or exercise during the past month (other than regular job)
- SMOKE100 — Smoked at least 100 cigarettes during lifetime
- _BMI5 — Body Mass Index (BMI)

---

## Potential Additional Variables
- LASTDEN4 — Time since last dental visit
- RMVTETH4 — Number of permanent teeth removed due to decay or gum disease

---

## Notes

These variables were selected based on their potential relationship to healthcare access barriers and delayed medical care due to cost. Final inclusion decisions will be made after evaluating missingness patterns, variable distributions, multicollinearity, and model performance.

In [166]:
# Candidate variables selected for initial review

candidate_vars = [

    "MEDCOST1",

    "SEXVAR",

    "MARITAL",

    "EDUCA",

    "RENTHOM1",

    "_AGE80",

    "_RACEGR3",

    "INCOME3",

    "EMPLOY1",

    "VETERAN3",

    "PRIMINS2",

    "PERSDOC3",

    "CHECKUP1",

    "GENHLTH",

    "PHYSHLTH",

    "MENTHLTH",

    "POORHLTH",

    "CVDINFR4",

    "CVDCRHD4",

    "CVDSTRK3",

    "ASTHMA3",

    "ASTHNOW",

    "CHCSCNC1",

    "CHCOCNC1",

    "CHCCOPD3",

    "ADDEPEV3",

    "CHCKDNY2",

    "EXERANY2",

    "SMOKE100",

    "_BMI5",

    "LASTDEN4",

    "RMVTETH4"

]

In [167]:
# Check whether all candidate variables exist in the dataset

missing_vars = [var for var in candidate_vars if var not in df.columns]

print("Missing variables:", missing_vars)

Missing variables: []


In [168]:
# Create candidate variable dataframe

candidate_df = df[candidate_vars].copy()

candidate_df.head()

,MEDCOST1,SEXVAR,MARITAL,EDUCA,RENTHOM1,_AGE80,_RACEGR3,INCOME3,EMPLOY1,VETERAN3,...,CHCSCNC1,CHCOCNC1,CHCCOPD3,ADDEPEV3,CHCKDNY2,EXERANY2,SMOKE100,_BMI5,LASTDEN4,RMVTETH4
0,2.0,2.0,3.0,4.0,1.0,78.0,1.0,99.0,7.0,2.0,...,1.0,2.0,2.0,2.0,2.0,1.0,2.0,2249.0,1.0,1.0
1,2.0,1.0,1.0,6.0,1.0,80.0,1.0,11.0,7.0,1.0,...,2.0,2.0,2.0,2.0,2.0,1.0,1.0,2583.0,1.0,1.0
2,1.0,1.0,6.0,5.0,1.0,59.0,1.0,99.0,1.0,1.0,...,2.0,2.0,2.0,2.0,2.0,1.0,1.0,2253.0,4.0,2.0
3,2.0,1.0,1.0,6.0,1.0,80.0,1.0,6.0,7.0,2.0,...,1.0,2.0,2.0,2.0,2.0,1.0,2.0,2509.0,1.0,8.0
4,2.0,1.0,5.0,5.0,1.0,47.0,1.0,3.0,8.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1977.0,1.0,1.0


In [169]:
# Missingness percentage for candidate variables

missing_pct = (

    candidate_df

    .isna()

    .mean()

    .mul(100)

    .sort_values(ascending=False)

)

missing_pct

ASTHNOW     84.260056
POORHLTH    41.388118
_BMI5        9.403500
SMOKE100     6.305854
INCOME3      2.025258
EMPLOY1      0.721699
VETERAN3     0.269845
MARITAL      0.001966
RENTHOM1     0.001748
EDUCA        0.001529
CHCCOPD3     0.001529
CHCKDNY2     0.001311
CHCSCNC1     0.001311
MEDCOST1     0.001311
PHYSHLTH     0.001092
GENHLTH      0.001092
ADDEPEV3     0.001092
CHCOCNC1     0.001092
RMVTETH4     0.001092
CVDCRHD4     0.000655
CVDSTRK3     0.000655
ASTHMA3      0.000655
MENTHLTH     0.000655
PERSDOC3     0.000655
PRIMINS2     0.000655
EXERANY2     0.000655
CVDINFR4     0.000437
CHECKUP1     0.000437
LASTDEN4     0.000437
SEXVAR       0.000000
_RACEGR3     0.000000
_AGE80       0.000000
dtype: float64

In [170]:
# Value counts for each candidate variable

for var in candidate_vars:
    print("\n" + "=" * 70)
    print(f"Variable: {var}")

    print(f"Unique Values: {df[var].nunique(dropna=False)}")

    print("\nValue Counts:")
    print(df[var].value_counts(dropna=False))

    print("\nMissing %:")
    print(round(df[var].isna().mean() * 100, 2))


Variable: MEDCOST1
Unique Values: 5

Value Counts:
MEDCOST1
2.0    412634
1.0     43363
7.0      1229
9.0       438
NaN         6
Name: count, dtype: int64

Missing %:
0.0

Variable: SEXVAR


Unique Values: 2

Value Counts:
SEXVAR
2.0    240183
1.0    217487
Name: count, dtype: int64

Missing %:
0.0

Variable: MARITAL
Unique Values: 8

Value Counts:
MARITAL
1.0    229393
5.0     84851
2.0     58827
3.0     49161
6.0     21754
4.0      9462
9.0      4213
NaN         9
Name: count, dtype: int64

Missing %:
0.0

Variable: EDUCA
Unique Values: 8

Value Counts:
EDUCA
6.0    191549
5.0    120732
4.0    115877
3.0     17200
2.0      9226
9.0      2356
1.0       723
NaN         7
Name: count, dtype: int64

Missing %:
0.0

Variable: RENTHOM1
Unique Values: 6

Value Counts:
RENTHOM1
1.0    312589
2.0    118460
3.0     22677
9.0      2835
7.0      1101
NaN         8
Name: count, dtype: int64

Missing %:
0.0

Variable: _AGE80
Unique Values: 63

Value Counts:
_AGE80
80.0    41770
70.0    10484
65.0    10386
67.0     9810
68.0     9483
        ...  
21.0     4290
22.0     4197
20.0     4159
19.0     4009
18.0     3773
Name: count, Length: 63, dtype: int64

Missing %:
0.0

Variable: _RACE

In [171]:
for var in candidate_vars:
    print("\n" + "=" * 70)
    print(f"Variable: {var}")
    print(f"Unique Values: {df[var].nunique(dropna=False)}")
    print(f"Missing %: {round(df[var].isna().mean() * 100, 2)}")

    if df[var].nunique(dropna=False) > 50:
        print("\nSummary Statistics:")
        print(df[var].describe())
    else:
        print("\nValue Counts:")
        print(df[var].value_counts(dropna=False))


Variable: MEDCOST1
Unique Values: 5
Missing %: 0.0

Value Counts:
MEDCOST1
2.0    412634
1.0     43363
7.0      1229
9.0       438
NaN         6
Name: count, dtype: int64

Variable: SEXVAR
Unique Values: 2
Missing %: 0.0

Value Counts:
SEXVAR
2.0    240183
1.0    217487
Name: count, dtype: int64

Variable: MARITAL
Unique Values: 8
Missing %: 0.0

Value Counts:
MARITAL
1.0    229393
5.0     84851
2.0     58827
3.0     49161
6.0     21754
4.0      9462
9.0      4213
NaN         9
Name: count, dtype: int64

Variable: EDUCA
Unique Values: 8
Missing %: 0.0

Value Counts:
EDUCA
6.0    191549
5.0    120732
4.0    115877
3.0     17200
2.0      9226
9.0      2356
1.0       723
NaN         7
Name: count, dtype: int64

Variable: RENTHOM1
Unique Values: 6
Missing %: 0.0

Value Counts:
RENTHOM1
1.0    312589
2.0    118460
3.0     22677
9.0      2835
7.0      1101
NaN         8
Name: count, dtype: int64

Variable: _AGE80
Unique Values: 63
Missing %: 0.0

Summary Statistics:
count    457670.000000
m

In [172]:
priority_vars = {
    "MEDCOST1": "Unable to see a doctor because of cost during the past 12 months (target variable)",
    "INCOME3": "Annual household income from all sources",
    "PRIMINS2": "Current primary source of health care coverage",
    "EMPLOY1": "Current employment status",
    "PERSDOC3": "Personal doctor or health care provider status",
    "CHECKUP1": "Time since last routine medical checkup",
    "GENHLTH": "Self-reported general health",
    "EDUCA": "Highest grade or year of school completed",
    "SEXVAR": "Sex of respondent",
    "MARITAL": "Marital status",
    "_AGE80": "Age in years",
    "_RACEGR3": "Race/ethnicity category"
}

In [173]:
secondary_vars = {
    "PHYSHLTH": "Number of days physical health was not good during the past 30 days",
    "MENTHLTH": "Number of days mental health was not good during the past 30 days",
    "RENTHOM1": "Home ownership status (own or rent)",
    "VETERAN3": "Ever served on active duty in the United States Armed Forces",
    "_BMI5": "Body Mass Index (BMI)",
    "SMOKE100": "Smoked at least 100 cigarettes during lifetime",
    "EXERANY2": "Participated in physical activities or exercise during the past month, other than regular job"
}

In [174]:
chronic_vars = {
    "CVDINFR4": "Ever told you had a heart attack (myocardial infarction)",
    "CVDCRHD4": "Ever told you had angina or coronary heart disease",
    "CVDSTRK3": "Ever told you had a stroke",
    "ASTHMA3": "Ever told you had asthma",
    "ASTHNOW": "Still have asthma",
    "CHCSCNC1": "Ever told you had non-melanoma skin cancer",
    "CHCOCNC1": "Ever told you had melanoma or another type of cancer",
    "CHCCOPD3": "Ever told you had COPD, emphysema, or chronic bronchitis",
    "ADDEPEV3": "Ever told you had a depressive disorder",
    "CHCKDNY2": "Ever told you had kidney disease (excluding kidney stones, bladder infections, or incontinence)"
}

In [175]:
# Priority Variable Review

In [176]:
for var, desc in priority_vars.items():
    print("\n" + "=" * 80)
    print(f"{var}: {desc}")
    print(df[var].value_counts(dropna=False))


MEDCOST1: Unable to see a doctor because of cost during the past 12 months (target variable)
MEDCOST1
2.0    412634
1.0     43363
7.0      1229
9.0       438
NaN         6
Name: count, dtype: int64

INCOME3: Annual household income from all sources
INCOME3
7.0     60709
9.0     55778
8.0     51700
6.0     49686
5.0     41702
99.0    41041
77.0    37113
11.0    29694
10.0    26675
4.0     19841
3.0     13662
2.0     10542
1.0     10258
NaN      9269
Name: count, dtype: int64

PRIMINS2: Current primary source of health care coverage
PRIMINS2
1.0     155360
3.0     145567
2.0      38611
5.0      32847
88.0     25406
7.0      16939
77.0     12524
9.0      12329
10.0     10128
99.0      6017
8.0       1409
4.0        386
6.0        144
NaN          3
Name: count, dtype: int64

EMPLOY1: Current employment status
EMPLOY1
1.0    186378
7.0    146916
2.0     39146
8.0     27758
5.0     17832
6.0     11473
4.0     10812
3.0      8953
9.0      5099
NaN      3303
Name: count, dtype: int64

PERSDO

In [177]:
[col for col in df.columns if "AGE" in col]

['DIABAGE4',
 'CNCRAGE',
 'CAGEG',
 '_AGEG5YR',
 '_AGE65YR',
 '_AGE80',
 '_AGE_G',
 '_LCSAGE']

In [178]:
[col for col in df.columns if "RACE" in col]

['_IMPRACE', '_CRACE1', '_MRACE1', '_RACE', '_RACEG21', '_RACEGR3', '_RACEPRV']

In [179]:
df["_AGE80"].describe()

count    457670.000000
mean         55.084784
std          18.127835
min          18.000000
25%          40.000000
50%          58.000000
75%          71.000000
max          80.000000
Name: _AGE80, dtype: float64

In [180]:
df["_AGE80"].value_counts(dropna=False).sort_index()

_AGE80
18.0     3773
19.0     4009
20.0     4159
21.0     4290
22.0     4197
        ...  
76.0     8109
77.0     8314
78.0     6395
79.0     5434
80.0    41770
Name: count, Length: 63, dtype: int64

In [181]:
age_vars = [
    "_AGE80",
    "_AGEG5YR",
    "_AGE65YR",
    "_AGE_G"
]

for var in age_vars:
    print("\n" + "="*50)
    print(var)
    print(df[var].value_counts(dropna=False))


_AGE80
_AGE80
80.0    41770
70.0    10484
65.0    10386
67.0     9810
68.0     9483
        ...  
21.0     4290
22.0     4197
20.0     4159
19.0     4009
18.0     3773
Name: count, Length: 63, dtype: int64

_AGEG5YR
_AGEG5YR
10.0    47701
11.0    44774
9.0     43387
13.0    41756
12.0    36803
8.0     34936
7.0     31698
5.0     30899
1.0     29692
6.0     28968
4.0     28804
3.0     26237
2.0     23705
14.0     8310
Name: count, dtype: int64

_AGE65YR
_AGE65YR
1.0    278326
2.0    171034
3.0      8310
Name: count, dtype: int64

_AGE_G
_AGE_G
6.0    172826
5.0     80280
4.0     64458
3.0     60418
2.0     49994
1.0     29694
Name: count, dtype: int64


# Age Variable Selection

Several BRFSS age variables were reviewed. _AGE80 was selected for
modeling because it preserves the greatest level of age detail,
providing individual ages through 79 years and a top-coded category
for respondents aged 80 and older.

In [182]:
df["_RACEGR3"].value_counts(dropna=False)

_RACEGR3
1.0    329346
5.0     48646
2.0     35172
3.0     24917
4.0     10486
9.0      9103
Name: count, dtype: int64

In [183]:
race_vars = [
    "_IMPRACE",
    "_CRACE1",
    "_MRACE1",
    "_RACE",
    "_RACEG21",
    "_RACEGR3",
    "_RACEPRV"
]

for var in race_vars:
    print("\n" + "="*70)
    print(var)
    print(df[var].value_counts(dropna=False))


_IMPRACE
_IMPRACE
1.0    336908
5.0     48961
2.0     35559
6.0     16861
3.0     12760
4.0      6621
Name: count, dtype: int64

_CRACE1
_CRACE1
NaN     399071
1.0      41113
2.0       4877
7.0       3479
99.0      2717
4.0       2132
6.0       1412
3.0       1398
77.0      1046
5.0        425
Name: count, dtype: int64

_MRACE1
_MRACE1
1.0     358959
2.0      40393
4.0      13084
7.0      11680
6.0      10261
3.0       9156
99.0      5886
77.0      5439
5.0       2807
NaN          5
Name: count, dtype: int64

_RACE
_RACE
1.0    329346
8.0     48646
2.0     35172
4.0     12646
7.0     10486
9.0      9103
3.0      6460
6.0      3737
5.0      2074
Name: count, dtype: int64

_RACEG21
_RACEG21
1.0    329346
2.0    119221
9.0      9103
Name: count, dtype: int64

_RACEGR3
_RACEGR3
1.0    329346
5.0     48646
2.0     35172
3.0     24917
4.0     10486
9.0      9103
Name: count, dtype: int64

_RACEPRV
_RACEPRV
1.0    336908
8.0     48961
2.0     35559
4.0     12760
7.0     10486
3.0      6621
6

# Race Variable Selection

Several BRFSS race and ethnicity variables were reviewed.

_RACEGR3 was selected because it provides five interpretable race/ethnicity categories while maintaining sufficient representation across groups for analysis. More detailed race variables contained additional categories that would likely require consolidation during modeling, while broader variables combined groups in ways that could mask meaningful differences in healthcare access outcomes.

In [184]:
pd.crosstab(
    df["ASTHMA3"],
    df["ASTHNOW"],
    dropna=False
)

ASTHNOW,1.0,2.0,7.0,9.0,NaN
ASTHMA3,,,,,
1.0,49694,20267,2019,57,0
2.0,0,0,0,0,383772
7.0,0,0,0,0,1650
9.0,0,0,0,0,208
NaN,0,0,0,0,3


## ASTHNOW Review

ASTHNOW is only asked of respondents who reported having asthma (ASTHMA3 = Yes). Therefore, the high percentage of missing values reflects BRFSS survey skip logic rather than respondent nonresponse.

Because ASTHMA3 already captures whether a respondent has ever been diagnosed with asthma, ASTHNOW may provide limited additional information while introducing additional preprocessing complexity. This variable will be evaluated further during feature selection and may be excluded from the final modeling dataset.

# Key EDA Findings and Next Steps

The 2024 BRFSS dataset contains 457,670 respondents and 301 variables. MEDCOST1 was confirmed as the supervised learning target variable.

The target variable is imbalanced: approximately 9.5% of respondents reported being unable to see a doctor because of cost, while approximately 90.2% reported no cost barrier. This means downstream model evaluation should not rely on accuracy alone.

All candidate variables selected for initial review were present in the dataset. Several BRFSS age and race variables were reviewed, and _AGE80 and _RACEGR3 were selected for the initial modeling dataset.

_AGE80 was selected because it preserves the greatest level of age detail available in the dataset, with individual ages through 79 and a top-coded category for respondents age 80 and older.

_RACEGR3 was selected because it provides five race/ethnicity categories while remaining easy to interpret. More detailed race variables contained additional categories that would likely require consolidation during modeling, while broader variables combined groups in ways that could mask meaningful differences in healthcare access outcomes.

Most variables have minimal true missingness, but ASTHNOW, POORHLTH, _BMI5, SMOKE100, and INCOME3 require additional review or preprocessing decisions. The ASTHMA3/ASTHNOW cross-tab suggests that ASTHNOW missingness is driven by BRFSS survey skip logic rather than respondent nonresponse.

Next steps:
1. Modularize feature lists into reusable functions.
2. Review special BRFSS response codes such as 7, 9, 77, 88, and 99.
3. Create recoded feature sets for modeling.
4. Document keep/drop/transform decisions for each candidate variable.
5. Create a cleaned modeling dataset for team handoff.